# Stock Ticker

In [5]:
!pip cache purge --quiet

In [6]:
import pandas as pd

In [7]:
# If you've forked this repo, change OWNER to your GitHub username.
# REPO and BRANCH will normally stay the same unless you renamed them.
OWNER = "singlestore-cookbook"
REPO = "singlestore-cookbook.github.io"
BRANCH = "refs/heads/main"

BASE_URL = f"https://raw.githubusercontent.com/{OWNER}/{REPO}/{BRANCH}/code/part-ml/running-sentiment-analysis-inside-the-database-with-webassembly/datasets"

In [8]:
tick_csv_url = f"{BASE_URL}/fictitious_stocks.csv"

tick_df = pd.read_csv(tick_csv_url)

In [9]:
tick_df = tick_df.dropna()

In [10]:
tick_df = tick_df[tick_df["volume"] <= 2_147_483_647]

In [11]:
tick_df.count()

date      379764
open      379764
high      379764
low       379764
close     379764
volume    379764
Name      379764
dtype: int64

In [12]:
tick_df = tick_df.rename(columns = {"date": "ts", "Name": "symbol"})
tick_df = tick_df.sort_values(by = ["ts", "symbol"])

In [13]:
tick_df.head()

,ts,open,high,low,close,volume,symbol
0,2013-01-02,743.98,756.93,736.15,745.68,9142645,BBRQ-FX
755,2013-01-02,418.92,426.82,415.64,420.77,2281501,BBYX-FX
1510,2013-01-02,192.05,192.73,190.29,192.03,194074,BFDS-FX
2265,2013-01-02,108.47,109.55,107.06,108.30,6371511,BGRP-FX
3020,2013-01-02,188.60,191.23,187.27,187.60,12854613,BJBY-FX


<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>Select the database from the drop-down menu at the top of this notebook. It updates the <b>connection_url</b> which is used by SQLAlchemy to make connections to the selected database.</p>
    </div>
</div>

In [14]:
from sqlalchemy import *

db_connection = create_engine(connection_url)

In [15]:
with db_connection.begin() as conn:
    conn.execute(text("TRUNCATE TABLE tick;"))

In [16]:
tick_df.to_sql(
    "tick",
    con = db_connection,
    if_exists = "append",
    index = False,
    chunksize = 1000
)

379764